<a href="https://colab.research.google.com/github/ibrahimymhafez/flyrank-ibrahim/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimymhafez/flyrank-ibrahim/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

# Safely get your Hugging Face token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Connect to DuckDB and pass the token
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Define the path to the mid-panel month partition
table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("DuckDB connected and ready to query the warehouse.")

DuckDB connected and ready to query the warehouse.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [56]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# We will define these fields in our SQL query in the next section.
# This cell is just to confirm our bucket lists programmatically.
features = ['impressions', 'clicks', 'ctr', 'is_weekend']
context = ['content_hash_id', 'client_hash_id', 'report_date']
label = 'needs_refresh'
excluded = ['future_clicks']

print("Contract fields defined.")

Contract fields defined.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fact 1: The Grain (Is one row really one content item per client per day?)
grain_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT concat(report_date, client_hash_id, content_hash_id)) as unique_grain_count
    FROM read_parquet('{table_path}')
""").df()
print("Fact 1 - Grain Match (Total Rows vs Unique Combinations):")
print(grain_check)

# Fact 2: Row Count and Date Span for our slice
span_check = con.sql(f"""
    SELECT
        COUNT(*) as row_count,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM read_parquet('{table_path}')
""").df()
print("\nFact 2 - Slice Dimensions:")
print(span_check)

# Fact 3: Availability & Feature Frame Building
# Filter where impressions are valid (IS TRUE) and build the frame
df_frame = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions as impressions,
        gsc_clicks as clicks,
        (gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as ctr,
        DAYOFWEEK(report_date) IN (0, 6) as is_weekend,
        (gsc_clicks = 0 AND gsc_impressions > 0) as needs_refresh
    FROM read_parquet('{table_path}')
    WHERE gsc_impressions IS NOT NULL
    LIMIT 10000 -- Keeping it small for local memory in this exercise
""").df()

surviving_rows = len(df_frame)
print(f"\nFact 3 - Availability: {surviving_rows} rows survive the IS NOT NULL filter.")
display(df_frame.head(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1 - Grain Match (Total Rows vs Unique Combinations):
   total_rows  unique_grain_count
0     9841378             9841378

Fact 2 - Slice Dimensions:
   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31

Fact 3 - Availability: 10000 rows survive the IS NOT NULL filter.


,content_hash_id,client_hash_id,report_date,impressions,clicks,ctr,is_weekend,needs_refresh
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,0.000,True,True
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000,True,True
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,0.008,True,False


In [55]:
# Fact 4: Windows - How many content items are 'new' (0-30 days old)?
# Calculate age of content for each row in df_frame
latest_report_date = df_frame['report_date'].max()
df_frame['age_in_days'] = (latest_report_date - df_frame['report_date']).dt.days

# Identify 'new' content (0-30 days old)
new_content_count = df_frame[df_frame['age_in_days'] <= 30]['content_hash_id'].nunique()

print(f"\nFact 4 - New Content Windows: {new_content_count} unique content items are 0-30 days old.")


Fact 4 - New Content Windows: 10000 unique content items are 0-30 days old.


## 4. Data limits
**What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.**



In [61]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

df_clean = df_frame.copy()

df_clean['ctr'] = df_clean['ctr'].fillna(0)

df_clean['leaky_metric'] = df_clean[label] * 0.95 + np.random.normal(0, 0.05, len(df_clean))

X_trap = df_clean[features + ['leaky_metric']]
y = df_clean[label]

model_trap = LogisticRegression(max_iter=1000)
model_trap.fit(X_trap, y)
trap_score = accuracy_score(y, model_trap.predict(X_trap))
print(f"Model score WITH the leaky feature (The Trap): {trap_score:.4f} (Dangerously perfect)")

X_honest = df_clean[features]
model_honest = LogisticRegression(max_iter=1000)
model_honest.fit(X_honest, y)
honest_score = accuracy_score(y, model_honest.predict(X_honest))
print(f"Model score AFTER deleting the leakage (Honest): {honest_score:.4f}")

Model score WITH the leaky feature (The Trap): 1.0000 (Dangerously perfect)
Model score AFTER deleting the leakage (Honest): 0.8356


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.